In [1]:
!nvidia-smi
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.155 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 7037.7/8062.4 GB disk)


In [2]:
import glob, os, shutil
from PIL import Image
from collections import Counter

src = "/kaggle/input/datasets/liuxiaolong1/pcb-defect-detection-dataset/DeepPCB"
dst = "/kaggle/working/pcb_dataset"

names = ['Missing_hole', 'Mouse_bite', 'Open_circuit', 'Short', 'Spur', 'Spurious_copper']

dist = Counter()
for split in ["train", "valid", "test"]:
    os.makedirs(f"{dst}/{split}/images", exist_ok=True)
    os.makedirs(f"{dst}/{split}/labels", exist_ok=True)

    for img_path in glob.glob(f"{src}/{split}/images/*"):
        fname = os.path.basename(img_path)
        shutil.copy(img_path, f"{dst}/{split}/images/{fname}")

        lbl_name = os.path.splitext(fname)[0] + ".txt"
        lbl_path = f"{src}/{split}/labels/{lbl_name}"
        if not os.path.exists(lbl_path):
            continue

        W, H = Image.open(img_path).size
        out = []
        for line in open(lbl_path):
            p = line.split()
            if len(p) < 5:
                continue
            x1, y1, x2, y2, cls = float(p[0]), float(p[1]), float(p[2]), float(p[3]), int(p[4])
            cls -= 1
            cx = ((x1 + x2) / 2) / W
            cy = ((y1 + y2) / 2) / H
            w  = (x2 - x1) / W
            h  = (y2 - y1) / H
            out.append(f"{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
            dist[names[cls]] += 1

        open(f"{dst}/{split}/labels/{lbl_name}", "w").write("\n".join(out))

print("✅ التحويل اكتمل")
for n in names:
    print(f"  {n:18s}: {dist[n]}")

yaml_content = f"""path: {dst}
train: train/images
val: valid/images
test: test/images

names: {names}
"""
open("/kaggle/working/data.yaml", "w").write(yaml_content)
print("✅ data.yaml جاهز")

✅ التحويل اكتمل
  Missing_hole      : 1942
  Mouse_bite        : 1506
  Open_circuit      : 1965
  Short             : 1625
  Spur              : 1474
  Spurious_copper   : 1501
✅ data.yaml جاهز


In [3]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

results = model.train(
    data="/kaggle/working/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    project="/kaggle/working/pcb_runs",
    name="train_v1",
    pretrained=True,
    seed=42,
)

Ultralytics 8.4.155 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train_v1, nbs=64, nms=None, opset=None, optimize=F

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


      2/100      5.07G      1.396       1.12      1.065        162        640: 100% ━━━━━━━━━━━━ 75/75 5.5it/s 13.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 4.4it/s 1.1s
                   all        150       1005      0.722      0.675      0.676      0.201

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      5.07G      1.302     0.9698      1.032        154        640: 100% ━━━━━━━━━━━━ 75/75 5.5it/s 13.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 4.6it/s 1.1s
                   all        150       1005      0.817      0.764      0.825      0.271

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100      5.07G      1.296     0.9394       1.03        159        640: 100% ━━━━━━━━━━━━ 75/75 5.4it/s 13.8s
                 Class     Images  Instances      Box(P        

In [4]:
best_model = YOLO("/kaggle/working/pcb_runs/train_v1/weights/best.pt")
metrics = best_model.val(data="/kaggle/working/data.yaml", split="test")

names = ['Missing_hole', 'Mouse_bite', 'Open_circuit', 'Short', 'Spur', 'Spurious_copper']

print("===== النتائج النهائية (Test) =====")
print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")
print(f"mAP@0.5   : {metrics.box.map50:.4f}")
print(f"mAP@0.5:95: {metrics.box.map:.4f}\n")

print("===== لكل فئة =====")
for i, name in enumerate(names):
    print(f"{name:15s} | P={metrics.box.p[i]:.3f} | R={metrics.box.r[i]:.3f} | mAP50={metrics.box.ap50[i]:.3f} | mAP50-95={metrics.box.maps[i]:.3f}")

Ultralytics 8.4.155 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 100 layers, 9,415,122 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 859.6±240.3 MB/s, size: 33.3 KB)
val: Scanning /kaggle/working/pcb_dataset/test/labels... 150 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 150/150 1.2Kit/s 0.1s
val: New cache created: /kaggle/working/pcb_dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 2.9it/s 3.4s
                   all        150        984      0.974      0.953      0.982      0.758
          Missing_hole        134        191       0.98      0.963      0.983      0.695
            Mouse_bite        114        170      0.975      0.909      0.955      0.687
          Open_circuit        129        194      0.977      0.948      0.991      0.745
                 Short        113        153      0.987      

In [5]:
best_model.predict(source="/kaggle/working/pcb_dataset/test/images", save=True, conf=0.25,
                   project="/kaggle/working/pcb_runs", name="predict_test")
print("✅ صور التنبؤ محفوظة")


image 1/150 /kaggle/working/pcb_dataset/test/images/01_PCB__101.jpg: 640x640 1 Missing_hole, 1 Mouse_bite, 4 Open_circuits, 1 Spur, 2 Spurious_coppers, 15.5ms
image 2/150 /kaggle/working/pcb_dataset/test/images/01_PCB__1019.jpg: 640x640 3 Missing_holes, 1 Mouse_bite, 1 Open_circuit, 1 Short, 1 Spurious_copper, 15.5ms
image 3/150 /kaggle/working/pcb_dataset/test/images/01_PCB__1044.jpg: 640x640 2 Missing_holes, 2 Mouse_bites, 2 Open_circuits, 1 Short, 1 Spur, 4 Spurious_coppers, 15.5ms
image 4/150 /kaggle/working/pcb_dataset/test/images/01_PCB__1047.jpg: 640x640 1 Missing_hole, 1 Mouse_bite, 2 Open_circuits, 1 Short, 1 Spur, 1 Spurious_copper, 15.5ms
image 5/150 /kaggle/working/pcb_dataset/test/images/01_PCB__1053.jpg: 640x640 2 Missing_holes, 1 Open_circuit, 1 Short, 1 Spur, 1 Spurious_copper, 15.5ms
image 6/150 /kaggle/working/pcb_dataset/test/images/01_PCB__1057.jpg: 640x640 1 Missing_hole, 1 Mouse_bite, 2 Open_circuits, 1 Spur, 1 Spurious_copper, 15.4ms
image 7/150 /kaggle/working/